## Imports and setup

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import glob

# Ensure plots are displayed inline in Jupyter
%matplotlib inline

## Graph Construction & Visualization Function

In [ ]:
def visualize_kg(dataset_name="WR18", split="train", context_size=3, max_edges=100):
    """
    Constructs and visualizes a subset of the Knowledge Graph.
    """
    kg = nx.DiGraph()  # Using DiGraph since relationships are directional

    # Find all relation CSVs for the specified split and context size
    file_pattern = (
        f"output/{dataset_name}/{split}/c{context_size}/*_c{context_size}.csv"
    )
    files = glob.glob(file_pattern)

    if not files:
        print(f"No files found for pattern: {file_pattern}")
        return

    edges_added = 0

    # Parse files and build the graph
    for file_path in files:
        if edges_added >= max_edges:
            break

        with open(file_path, "r") as f:
            next(f)  # Skip header
            for line in f:
                parts = line.strip().split(";")
                if len(parts) < 2:
                    continue

                triple_str = parts[0]
                label = int(parts[1])

                # Only add verified facts (label == 1) to the graph
                if label == 1:
                    try:
                        subject, relation, obj = triple_str.split(",")
                        kg.add_edge(subject, obj, label=relation)
                        edges_added += 1

                        if edges_added >= max_edges:
                            break
                    except ValueError:
                        continue  # Skip malformed triples

    print(
        f"Constructed graph with {kg.number_of_nodes()} nodes and {kg.number_of_edges()} edges."
    )

    # Visualization
    plt.figure(figsize=(14, 10))

    # Use a spring layout for better node spacing
    pos = nx.spring_layout(kg, seed=42, k=0.5)

    # Draw nodes
    nx.draw_networkx_nodes(
        kg, pos, node_color="lightblue", node_size=2000, alpha=0.8, edgecolors="gray"
    )

    # Draw edges
    nx.draw_networkx_edges(
        kg, pos, edge_color="gray", arrows=True, arrowsize=20, width=1.5
    )

    # Draw node labels (subject/object names)
    nx.draw_networkx_labels(kg, pos, font_size=10, font_weight="bold")

    # Draw edge labels (relations)
    edge_labels = nx.get_edge_attributes(kg, "label")
    nx.draw_networkx_edge_labels(
        kg, pos, edge_labels=edge_labels, font_color="red", font_size=8
    )

    plt.title(
        f"SciCheck Knowledge Graph Subset ({split.capitalize()} split)", fontsize=16
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## Execute the visualization

In [ ]:
# Visualizing the ground truth facts from the training set
visualize_kg(dataset_name="WN11", split="train", context_size=3, max_edges=50)

No files found for pattern: output/WR18/train/c3/*_c3.csv
